## Import module + Function

In [ ]:
import os

import duckdb
from dotenv import load_dotenv

load_dotenv()

In [ ]:
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    SET s3_region='{os.environ["AWS_REGION"]}';
    SET s3_access_key_id='{os.environ["AWS_ACCESS_KEY_ID"]}';
    SET s3_secret_access_key='{os.environ["AWS_SECRET_ACCESS_KEY"]}';
""")

## Bronze layer

In [3]:
BRONZE_URI = "s3://finance-transaction-datalake-dev/bronze/"

#### Cards

In [4]:
CARD_URI = BRONZE_URI + "cards_data.csv"

In [5]:
cards_df = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{CARD_URI}')
    LIMIT 10
""").fetchdf()

In [6]:
cards_df

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,True,2,$24295,09/2002,2008,False
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,True,2,$21968,04/2014,2014,False
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,True,2,$46414,07/2003,2004,False
3,42,825,Visa,Credit,4879494103069057,08/2024,693,False,1,$12400,01/2003,2012,False
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,True,1,$28,09/2008,2009,False
5,4537,1746,Visa,Credit,4404898874682993,09/2003,736,True,1,$27500,09/2003,2012,False
6,1278,1746,Visa,Debit,4001482973848631,07/2022,972,True,2,$28508,02/2011,2011,False
7,3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,True,2,$9022,07/2003,2015,False
8,3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,True,2,$54,06/2010,2015,False
9,3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,True,1,$99,07/2006,2012,False


#### Users

In [7]:
USER_URI = BRONZE_URI + "users_data.csv"

In [8]:
users_df = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{USER_URI}')
    LIMIT 8
""").fetchdf()

In [9]:
users_df

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
5,68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.60,$20599,$41997,$0,704,3
6,1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
7,1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1


#### Transactions

In [10]:
TRANS_URI = BRONZE_URI + "transactions_data.csv"

In [11]:
trans_df = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{TRANS_URI}')
    LIMIT 8
""").fetchdf()

In [12]:
trans_df

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,None
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,None
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,None
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,None
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,None
5,7475333,2010-01-01 00:07:00,1807,165,$4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,None
6,7475334,2010-01-01 00:09:00,1556,2972,$77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,None
7,7475335,2010-01-01 00:14:00,1684,2140,$26.46,Online Transaction,39021,ONLINE,NaN,NaN,4784,None


## Silver layer

In [13]:
SILVER_URI="s3://finance-transaction-datalake-dev/silver/"

#### Users

In [15]:
USER_S_URI = SILVER_URI + "users/part-00000-3767a367-7218-48cb-b853-1faf22976126-c000.snappy.parquet"

user_s_uri = con.execute(f"""
    SELECT *
    FROM read_parquet('{USER_S_URI}')
    LIMIT 8
""").fetchdf()

In [16]:
user_s_uri

,user_id,retirement_age,birth_year,birth_month,gender,city,state,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted
0,0,69,1986,3,MALE,Scarborough,Maine,29237.0,59613.0,36199.0,763,4,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
1,1,74,1976,4,FEMALE,East Pensacola Heights,Florida,22247.0,45360.0,14587.0,704,3,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
2,10,66,1990,2,MALE,Miami,Florida,28871.0,58865.0,94134.0,727,2,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
3,100,66,1963,9,MALE,High Point,North Carolina,24005.0,48944.0,79960.0,813,7,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
4,1000,73,1999,8,FEMALE,Attleboro,Massachusetts,26693.0,54424.0,92199.0,687,1,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
5,1001,62,1999,10,MALE,Fairmount,Tennessee,17795.0,36283.0,60989.0,716,1,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
6,1002,67,1988,4,MALE,Oildale,California,19023.0,38788.0,77448.0,716,1,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False
7,1003,62,1968,11,MALE,Morrisville,Pennsylvania,39495.0,80526.0,117380.0,632,1,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False


In [17]:
user_825 = con.execute(f"""
    SELECT *
    FROM read_parquet('{USER_S_URI}')
    WHERE user_id = 825
""").fetchdf()

user_825

,user_id,retirement_age,birth_year,birth_month,gender,city,state,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted
0,825,66,1966,11,FEMALE,La Verne,California,29278.0,59696.0,127613.0,787,5,2026-06-15 14:14:54.745487,,00g6fq2qnqb8mo27,2026-06-15 14:14:54.745487,2036-01-01,False


#### Transaction

In [18]:
# year 2010, month 1, day 1
TRANS_S_URI = SILVER_URI + "transactions/year=2010/month=1/day=1/part-00058-d76bfd35-f22a-4929-a553-272badcab3a0.c000.snappy.parquet"

In [19]:
trans_s_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{TRANS_S_URI}')
    LIMIT 8
""").fetchdf()

trans_s_df

,transaction_id,timestamp,client_id,card_id,amount,transaction_channel,merchant_id,merchant_city,merchant_state,zip,...,errors,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted,day,month,year
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,SWIPE TRANSACTION,59935,beulah,ND,58523,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
1,7475328,2010-01-01 00:02:00,561,4575,14.57,SWIPE TRANSACTION,67570,bettendorf,IA,52722,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
2,7475329,2010-01-01 00:02:00,1129,102,80.00,SWIPE TRANSACTION,27092,vista,CA,92084,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
3,7475331,2010-01-01 00:05:00,430,2860,200.00,SWIPE TRANSACTION,27092,crown point,IN,46307,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
4,7475332,2010-01-01 00:06:00,848,3915,46.41,SWIPE TRANSACTION,13051,harwood,MD,20776,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
5,7475333,2010-01-01 00:07:00,1807,165,4.81,SWIPE TRANSACTION,20519,bronx,NY,10464,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
6,7475334,2010-01-01 00:09:00,1556,2972,77.00,SWIPE TRANSACTION,59935,beulah,ND,58523,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010
7,7475335,2010-01-01 00:14:00,1684,2140,26.46,ONLINE TRANSACTION,39021,online,NaN,NaN,...,<NA>,2026-06-15 14:12:52.617396,,00g6fq2qnqb8mo27,2026-06-15 14:12:52.617396,2036-01-01,False,1,1,2010


#### Cards

In [21]:
CARDS_S_URI = SILVER_URI + "cards/part-00000-dae53499-13b4-4abc-bc6a-ffbc196b2007-c000.snappy.parquet"

cards_s_uri = con.execute(f"""
    SELECT *
    FROM read_parquet('{CARDS_S_URI}')
    LIMIT 8
""").fetchdf()

cards_s_uri

,card_id,client_id,card_brand,card_type,mask_card_number,expires,has_a_cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted
0,0,1362,AMEX,Credit,***********8401,2024-04-01,True,<NA>,2,33900.0,1991-01-01,2014,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
1,1,550,MASTERCARD,Credit,************2292,2024-06-01,True,<NA>,1,11600.0,1994-01-01,2013,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
2,10,1783,MASTERCARD,Debit,************7350,2023-12-01,True,<NA>,1,16597.0,1999-01-01,2009,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
3,100,1822,MASTERCARD,Debit,************9682,2021-09-01,True,<NA>,2,14133.0,2006-01-01,2010,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
4,1000,1786,MASTERCARD,Debit,************3893,2024-09-01,True,<NA>,1,8273.0,2002-02-01,2009,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
5,1001,59,MASTERCARD,Credit,************0036,2021-06-01,True,<NA>,2,8800.0,2002-02-01,2013,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
6,1002,1516,MASTERCARD,Debit,************5365,2020-04-01,True,<NA>,1,47153.0,2002-02-01,2006,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
7,1003,1449,MASTERCARD,Debit,************6101,2023-06-01,True,<NA>,2,1563.0,2002-02-01,2011,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False


#### MCC

In [22]:
MCC_S_URI = SILVER_URI + "mcc/part-00000-dfec0d44-b68b-4fee-a50a-e6dee0d7b1ae-c000.snappy.parquet"

mcc_s_uri = con.execute(f"""
    SELECT *
    FROM read_parquet('{MCC_S_URI}')
    LIMIT 8
""").fetchdf()

mcc_s_uri

,mcc_code,merchant_name,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted
0,1711,"Heating, Plumbing, Air Conditioning Contractors",2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
1,3000,Steelworks,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
2,3001,Steel Products Manufacturing,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
3,3005,Miscellaneous Metal Fabrication,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
4,3006,Miscellaneous Fabricated Metal Products,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
5,3007,Coated and Laminated Products,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
6,3008,Steel Drums and Barrels,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False
7,3009,Fabricated Structural Metal Products,2026-06-15 14:15:00.128614,,00g6fq2qnqb8mo27,2026-06-15 14:15:00.128614,2036-01-01,False


In [25]:
cards_df = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{CARD_URI}')
    WHERE id = 1
""").fetchdf()

cards_df

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1,550,Mastercard,Credit,5278231764792292,06/2024,396,True,1,$11600,01/1994,2013,False


In [26]:
cards_s_uri = con.execute(f"""
    SELECT *
    FROM read_parquet('{CARDS_S_URI}')
    WHERE card_id = 1
""").fetchdf()

cards_s_uri

,card_id,client_id,card_brand,card_type,mask_card_number,expires,has_a_cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,_created_at,_source_file,_processing_id,_updated_at,_batch_logical_date,_is_deleted
0,1,550,MASTERCARD,Credit,************2292,2024-06-01,True,<NA>,1,11600.0,1994-01-01,2013,<NA>,2026-06-15 14:14:40.071331,,00g6fq2qnqb8mo27,2026-06-15 14:14:40.071331,2036-01-01,False
